In [43]:
import os
from pathlib import Path
from kedro.framework.startup import bootstrap_project
from kedro.framework.session import KedroSession

# 1. Move to project root if we are in the notebooks folder
if Path.cwd().name == "notebooks":
    os.chdir("..")

# 2. Initialize Kedro
project_path = Path.cwd()
bootstrap_project(project_path)

# 3. Create session and get catalog
session = KedroSession.create(project_path)
context = session.load_context()
catalog = context.catalog

print(f"✅ Kedro context loaded! Project root: {project_path}")

[04/21/26 22:34:11] INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=740959;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro_telemetry\plugin.py\plugin.py]8;;\:]8;id=197839;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro_telemetry\plugin.py#242\242]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

✅ Kedro context loaded! Project root: g:\Unidades compartidas\Alianzas\3. Data\UNIS\unis-perm-flow


In [44]:
# 2. Importar todas las funciones del archivo nodes.py
import unis_perm_flow.pipelines.data_processing.nodes as nodes

# Esto te dará una lista limpia de todo lo definido en ese archivo de nodos
print(dir(nodes))

['Tuple', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '__warningregistry__', 'check_and_export_duplicates', 'check_dataframe', 'clean_column_names', 'clean_column_objects', 'clean_nulls', 'convert_all_standardized_dates', 'convert_standardized_dates', 'datetime', 'duckdb', 'np', 'numeric_conversion_node', 'pd', 're', 'remove_accents', 'remove_accents_and_special_chars', 'select_columns', 'unicodedata', 'unis_preprocessing_calaca', 'unis_preprocessing_estaca']


In [45]:
parameters = catalog.load('parameters')

[04/21/26 22:34:12] INFO     Loading data from parameters (MemoryDataset)...                   ]8;id=459418;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=867842;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

In [46]:
# Base de central con fechas de bajas, fechas de graduación
unis_estaca = catalog.load('unis_estaca')

                    INFO     Loading data from unis_estaca (ExcelDataset)...                   ]8;id=261179;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=691009;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

In [47]:
unis_estaca_sd, unis_estaca_cd = nodes.unis_preprocessing_estaca(
    unis_estaca = unis_estaca,
    unis_col_fechas = parameters['data_processing_estaca']['unis_col_fechas'],
    unis_col_emails =  parameters['data_processing_estaca']['unis_col_emails'],
    unis_col_dd = parameters['data_processing_estaca']['unis_col_dd'],
    unis_col_sort= parameters['data_processing_estaca']['unis_col_sort'],
    unis_niveles_academicos = parameters['data_processing_estaca']['unis_niveles_academicos'],
    unis_estaca_column_order = parameters['data_processing_estaca']['unis_estaca_column_order']
) 

## Activos

In [48]:
unis_estaca_sd_activos = unis_estaca_sd[unis_estaca_sd['estado'] == 'activo'].copy()

In [49]:
unis_estaca_sd_activos['cohorte'].value_counts().reset_index().sort_values(by='cohorte')

,cohorte,count
1,2025-01-14,197
4,2025-05-13,119
2,2025-09-02,167
5,2025-10-28,67
0,2026-01-13,215
3,2026-03-10,166


# Calendario

In [50]:
# Base de central con fechas de bajas, fechas de graduación
unis_calaca = catalog.load('unis_calaca')


[04/21/26 22:34:16] INFO     Loading data from unis_calaca (ExcelDataset)...                   ]8;id=413065;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=935484;file://c:\Users\User\miniconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

In [51]:
unis_calaca.columns


Index(['Cohorte', 'Periodo_Raw', 'Periodo_Inicial', 'Periodo_Actual', 'Sesion',
       'Semana', 'Fecha_Inicio', 'Fecha_Fin', 'Tipo'],
      dtype='str')

In [52]:
parameters['data_processing_calaca']


{
    'unis_calaca_col_fechas': ['fecha_inicio', 'fecha_fin'],
    'unis_calaca_col_fechaingreso': 'fecha_ingreso',
    'unis_calaca_col_fecha_inicial_semana': 'fecha_inicio',
    'unis_calaca_col_fecha_final_semana': 'fecha_fin',
    'unis_calaca_col_cohorte_inicial': 'periodo_inicial',
    'unis_calaca_col_sort': ['periodo_inicial', 'periodo_actual', 'fecha_inicio', 'semana'],
    'unis_calalaca_student_journey': ['onboarding', 'q1', 'qa'],
    'unis_calaca_journey_thresholds': [2, 4],
    'unis_calaca_column_order': [
        'periodo_raw',
        'cohorte',
        'periodo_inicial',
        'fecha_ingreso',
        'periodo_actual',
        'sesion',
        'fecha_inicio',
        'fecha_fin',
        'shifted_fecha_inicio',
        'semana',
        'semana_acumulada',
        'month',
        'mes_academico',
        'anio_gregoriano',
        'mes_gregoriano',
        'student_journey',
        'tipo'
    ]
}

In [53]:
unis_calaca.columns


Index(['Cohorte', 'Periodo_Raw', 'Periodo_Inicial', 'Periodo_Actual', 'Sesion',
       'Semana', 'Fecha_Inicio', 'Fecha_Fin', 'Tipo'],
      dtype='str')

In [54]:
nodes.unis_preprocessing_calaca(
    unis_calendario = unis_calaca,
    col_fechas = parameters['data_processing_calaca']['unis_calaca_col_fechas'],
    col_fechaingreso = parameters['data_processing_calaca']['unis_calaca_col_fechaingreso'],
    col_fecha_ini_sem = parameters['data_processing_calaca']['unis_calaca_col_fecha_inicial_semana'],
    col_fecha_fin_sem = parameters['data_processing_calaca']['unis_calaca_col_fecha_final_semana'],
    col_cohorte_ini = parameters['data_processing_calaca']['unis_calaca_col_cohorte_inicial'],
    col_sort = parameters['data_processing_calaca']['unis_calaca_col_sort'],
    journey_labels = parameters['data_processing_calaca']['unis_calalaca_student_journey'],
    journey_thresholds = parameters['data_processing_calaca']['unis_calaca_journey_thresholds'],
    col_ordenadas = parameters['data_processing_calaca']['unis_calaca_column_order']
)

[04/21/26 22:34:17] WARNING  G:\Unidades compartidas\Alianzas\3.                                    ]8;id=335443;file://c:\Users\User\miniconda3\envs\unis\Lib\warnings.py\warnings.py]8;;\:]8;id=106489;file://c:\Users\User\miniconda3\envs\unis\Lib\warnings.py#110\110]8;;\
                             Data\UNIS\unis-perm-flow\src\unis_perm_flow\pipelines\data_processing\                
                             nodes.py:109: UserWarning: Parsing dates in %d/%m/%Y format when                      
                             dayfirst=False (the default) was specified. Pass `dayfirst=True` or                   
                             specify a format to silence this warning.                                             
                               df = pd.to_datetime(df, errors='coerce')                                            
                                                                                                                   


(
                                            periodo_raw         cohorte  \
0    3er cuatrimestre 2024 9243 primer cuatrimestre       cohorte 1   
1    3er cuatrimestre 2024 9243 primer cuatrimestre       cohorte 1   
2    3er cuatrimestre 2024 9243 primer cuatrimestre       cohorte 1   
3    3er cuatrimestre 2024 9243 primer cuatrimestre       cohorte 1   
4    3er cuatrimestre 2024 9243 primer cuatrimestre       cohorte 1   
..                                              ...             ...   
955  5to cuatrimestre 2026 9275 cuarto cuatrimestre       cohorte 8   
956  5to cuatrimestre 2026 9275 cuarto cuatrimestre  cohorte 4 2026   
957  5to cuatrimestre 2026 9275 cuarto cuatrimestre       cohorte 8   
958  5to cuatrimestre 2026 9275 cuarto cuatrimestre       cohorte 8   
959  5to cuatrimestre 2026 9275 cuarto cuatrimestre  cohorte 4 2026   

     periodo_inicial fecha_ingreso  periodo_actual         sesion  \
0               9243    2024-08-27            9243  sesion 1 ct 1   
1 